In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
import numpy as np
import joblib

In [5]:
# Definition of attributes and targets

# =========================== 21 atributos seleccionados =========================== #

selected_21_attributes = [
    'np_without_modification', 'surface_modification', 'size_tem', 'zeta_potential', 
    'incubation_protein_source', 'incubation_plasma_concentration', 'incubation_np_concentration',
    'centrifugation_speed', 'centrifugation_time', 'centrifugation_temperature', 'np_type',
    'np_shape', 'dispersion_medium', 'dispersion_medium_ph', 'size_dls', 'pdi', 'incubation_culture',
    'incubation_time', 'incubation_temperature', 'centrifugation_repetitions', 'modification_type'
]


categorical_21_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
    'np_type', 'np_shape', 'dispersion_medium', 'incubation_culture', 'modification_type'
]

# =========================== 10 atributos seleccionados =========================== #

selected_10_attributes = [
    'np_without_modification', 'surface_modification', 
    'incubation_protein_source', 'size_tem', 'zeta_potential', 'incubation_plasma_concentration', 
    'incubation_np_concentration','centrifugation_speed', 
    'centrifugation_time', 'centrifugation_temperature'    
]

categorical_10_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
]

# =========================== 17 atributos seleccionados (Jair) =========================== #

selected_17_attributes = [
    'np_without_modification', 'surface_modification', 'zeta_potential', 
    'incubation_protein_source', 'incubation_plasma_concentration', 'incubation_np_concentration',
    'np_type', 'np_shape', 'dispersion_medium', 'dispersion_medium_ph', 'size_dls', 'pdi', 
    'incubation_culture', 'incubation_time', 'incubation_temperature', 'modification_type'
]

categorical_17_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
    'np_type', 'np_shape', 'dispersion_medium', 'incubation_culture', 'modification_type'
]

# =========================== 16 atributos seleccionados (sin np_type para el entrenamiento) =========================== #

selected_16_attributes = [
    'np_without_modification', 'surface_modification', 'zeta_potential', 
    'incubation_protein_source', 'incubation_plasma_concentration', 'incubation_np_concentration', 
    'np_shape', 'dispersion_medium', 'dispersion_medium_ph', 'size_dls', 'pdi', 
    'incubation_culture', 'incubation_time', 'incubation_temperature', 'modification_type'
]

categorical_16_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
    'np_shape', 'dispersion_medium', 'incubation_culture', 'modification_type'
]

# =========================== 178 Proteinas =========================== #

protein_targets = [
    "P01871", "P01024", "P02647", "P02649", "P02768", "P04004", "P01834", 
    "P10909", "P02652", "P00734", "P01009", "P01042", "P01857", "P01859", 
    "P01023", "P01619", "P04196", "P02656", "P04114", "P02787", "P02766", 
    "P06396", "P06727", "P0C0L5", "P02671", "P08603", "P02765", "Q14624", 
    "P0DOY2", "P01008", "P68871", "P01764", "P04003", "P0C0L4", "P01876", 
    "P02749", "P02675", "P01860", "P01011", "P00747", "P02774", "P00751", 
    "P19823", "P08697", "P19827", "P69905", "P02760", "P02748", "P02679", 
    "P02790", "B9A064", "P07996", "P05155", "P01766", "P12259", "P00738",
    "P02751", "P02654", "P04406", "P00739", "P02655", "P00736", "P07225", 
    "P05154", "P09871", "P60709", "P03952", "P02747", "P05546", "P02746",
    "P27169", "P01019", "P35542", "P04217", "Q14520", "P02743", "P02763", 
    "P49908", "O43866", "P18428", "P55056", "P04264", "P00748", "P01591", 
    "P01861", "Q92954", "P02776", "P0DJI8", "P01031", "P05156", "P13671", 
    "Q03591", "P06312", "P00740", "P01615", "P00742", "P04070", "O14791", 
    "P05090", "P02745", "P00488", "Q13103", "P01877", "P01700", "P10643", 
    "Q9UK55", "P03951", "Q06033", "Q96IY4", "P13645", "P04433", "P07358",
    "P27918", "P05452", "P20851", "P07357", "P07360", "P01599", "O95445", 
    "Q9BXR6", "P02775", "P02741", "P35579", "P15169", "Q13790", "P35858", 
    "P49747", "P36955", "P19652", "P07737", "P22891", "P35908", "P80748", 
    "P08514", "P01593", "Q04756", "P06733", "P23528", "P63104", "P18065", 
    "P08519", "Q86UX7", "P02753", "Q5TB80", "P22352", "P00746", "P35443", 
    "P62937", "P10720", "P48740", "P25311", "P43652", "P35527", "Q9Y490", 
    "P05106", "P17936", "P18206", "P02788", "P06702", "P01701", "Q6Q788", 
    "Q96PD5", "Q9UGM5", "O00391", "P11142", "P14618", "P23142", "P68366",
    "P12814", "P61224", "Q13201", "P67936", "P0DOY3", "P08709", "P01034", 
    "P81605", "P02533", "P11226"]

In [6]:
def pearson_r2(y_true, y_pred):
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return 0.0
    # Calculamos Pearson R y lo elevamos al cuadrado
    return np.corrcoef(y_true, y_pred)[0, 1] ** 2

In [9]:
def buscar_hiperparametros_optimos(csv_filename, selected_attributes, categorical_attributes, protein_targets):
    """
    Realiza una búsqueda exhaustiva (GridSearchCV) para encontrar los mejores 
    hiperparámetros de un Random Forest para cada proteína objetivo.
    Muestra los resultados en pantalla sin guardar modelos.
    """
    if not all([selected_attributes, categorical_attributes, protein_targets]):
        print("Error: Faltan especificar atributos, variables categóricas o targets.")
        return

    # 1. Carga y Preprocesamiento
    df = pd.read_csv(f"{csv_filename}.csv")
    X = df[selected_attributes]
    X = pd.get_dummies(X, columns=categorical_attributes)
    
    # 2. Definición de la grilla de hiperparámetros (Pensada para tu dataset de ~122 filas)
    param_grid = {
        'n_estimators': [200, 500],           # Cantidad de árboles
        'max_samples': [None, 0.3, 0.7, 0.9],      # Fracción de datos por árbol (None = 100% con reemplazo)
        'max_features': ['sqrt', 1.0],        # Variables a mirar en cada corte
        'max_depth': [None, 5, 10],           # Profundidad máxima del árbol
        'min_samples_leaf': [1, 2]            # Mínimo de muestras por hoja final
    }

    # Creamos el "Scorer" (Le decimos que un número más alto es mejor)
    pearson_scorer = make_scorer(pearson_r2, greater_is_better=True)
    
    total_combinaciones = (len(param_grid['n_estimators']) * len(param_grid['max_samples']) * len(param_grid['max_features']) * len(param_grid['max_depth']) * len(param_grid['min_samples_leaf']))
                           
    print(f"Iniciando GridSearchCV...")
    print(f"Evaluando {total_combinaciones} combinaciones (x 5 folds) por cada proteína.\n")

    mejores_resultados = {}

    # 3. Búsqueda
    for target in protein_targets:
        print(f"Buscando el óptimo para la proteína: {target} ... ", end="")
        y = df[target].values
        
        # Modelo base
        rf_base = RandomForestRegressor(random_state=42)
        
        # Configuración del buscador
        grid_search = GridSearchCV(
            estimator=rf_base,
            param_grid=param_grid,
            cv=5,                              
            scoring={
                'Pearson_R2': pearson_scorer,
                'RMSE': 'neg_root_mean_squared_error'
            },
            # ¡CLAVE! Le decimos que el ganador absoluto se elija en base al Pearson_R2
            refit='Pearson_R2',
            n_jobs=-1,  # Usa todos los núcleos de tu PC para que sea mucho más rápido
            verbose=0                          
        )
        
        # Ejecutar búsqueda
        grid_search.fit(X, y)
        
        # Guardar y mostrar resultados
        best_params = grid_search.best_params_
        mejores_resultados[target] = best_params
        
        print("¡Listo!")
        print(f"  -> Mejores parámetros: {best_params}\n")
        
    print("========================================================================")
    print("Búsqueda finalizada con éxito.")
    
    return mejores_resultados

In [ ]:
# Definimos en donde se encuentran los Datasets para entrenar los modelos
csv_filename="liposomes_dataset-np_type"
csv_directory="datasets"
csv_filepath=os.path.join("..", csv_directory, csv_directory)

# Definimos el nombre del archivo en donde se almacenaran los parametros optimos
output_filename='hiperparam_opt_lip-np_type.joblib'

# Llamada a la función (asumiendo que ya tenés definidas tus variables)
resultados_optimos = buscar_hiperparametros_optimos(
    csv_filename="./datasets/liposomes_dataset-np_type", 
    selected_attributes=selected_16_attributes, 
    categorical_attributes=categorical_16_attributes, 
    protein_targets=protein_targets
)

# Para GUARDAR la variable en tu disco:
joblib.dump(resultados_optimos, output_filename)
print("¡Diccionario guardado con éxito!")

Iniciando GridSearchCV...
Evaluando 96 combinaciones (x 5 folds) por cada proteína.

Buscando el óptimo para la proteína: P01871 ... ¡Listo!
  -> Mejores parámetros: {'max_depth': 10, 'max_features': 'sqrt', 'max_samples': None, 'min_samples_leaf': 1, 'n_estimators': 200}

Buscando el óptimo para la proteína: P01024 ... ¡Listo!
  -> Mejores parámetros: {'max_depth': 10, 'max_features': 'sqrt', 'max_samples': 0.3, 'min_samples_leaf': 1, 'n_estimators': 200}

Buscando el óptimo para la proteína: P02647 ... ¡Listo!
  -> Mejores parámetros: {'max_depth': 5, 'max_features': 1.0, 'max_samples': 0.3, 'min_samples_leaf': 2, 'n_estimators': 500}

Buscando el óptimo para la proteína: P02649 ... 

In [7]:
import joblib

# Para GUARDAR la variable en tu disco:
joblib.dump(resultados_optimos, 'hiperparam_opt_lip-np_type.joblib')
print("¡Diccionario guardado con éxito!")

# Para CARGARLA otro día que abras el Notebook:
resultados_optimos = joblib.load('hiperparam_opt_lip-np_type.joblib')

¡Diccionario guardado con éxito!
